# nb11.1 - Abstract Builder: the 250-Word Ceiling and the Five Required Moves

*Week 11, Chapter 11.1 companion. Maya's advisor sent the Monday email she had been dreading:
"Send me the new abstract by Friday." Her February draft is 347 words, calls the design
"a difference-in-differences approach to identify the effect," and ends on "we discuss
implications for policy." Three problems, one notebook.*

The reveal-the-trick version of the lesson: **a good empirical abstract is a contract with five
clauses, written in roughly fifty words apiece, totaling no more than 250 words.** The five clauses
are *Question*, *Setting*, *Design*, *Finding*, *Why-It-Matters*, in that order. Most rejected
abstracts fail not because the underlying paper is weak but because the abstract drifts between
clauses, buries the finding inside the design, or skips the why-it-matters and ends on a
methodological note. The fix is mechanical: a tokenizer that counts words against the 250-word
ceiling, plus a rule-based scorer that flags whether each of the five required moves is present
and whether the verb in the *Question* sentence matches the design.

By the end of this notebook you will have a single function `score_abstract(text)` that takes raw
abstract prose, returns a budget report and a per-move diagnosis, and prints a one-page card you
can paste into your manuscript margin. We will tune it on Maya's two drafts (bad, good) and check
that it generalizes to Devon's, Priya's, Sam's, and Leah's abstracts as well.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless: render to buffers, never to a screen

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)  # one pinned generator drives the whole notebook
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

## 1. The tokenizer: what counts as a word?

Word counting sounds trivial. It is not. Journals disagree about whether a hyphenated
compound (`Black-White`) counts as one word or two, about whether numbers and units (`1.4
percentage points`) count as one token or three, and about whether `Callaway-Sant'Anna`
counts as a single citation token or a multi-word author string. Our rule, chosen for
*conservatism* (it tends to overcount, so an abstract that passes our tokenizer will pass
every journal's tokenizer): split on whitespace after stripping LaTeX commands and citation
brackets, and keep every nonempty resulting token. Hyphens stay inside the token. This
matches what most word-processor counters do and is the convention the Journal of Finance,
RFS, and JFE all use as of the 2024 author guidelines.

In [ ]:
def count_words(text):
    # Strip LaTeX commands (\command or \command{...}), bracket citations [..], and parens citations.
    t = re.sub(r"\\[a-zA-Z]+\{[^}]*\}", " ", text)   # \textit{foo}
    t = re.sub(r"\\[a-zA-Z]+", " ", t)                 # \alpha
    t = re.sub(r"\$[^$]*\$", " MATH ", t)              # inline math -> single token
    t = re.sub(r"\([^)]*\d{4}[^)]*\)", " CITE ", t)    # (Author 2021) -> CITE
    tokens = [w for w in re.split(r"\s+", t.strip()) if w]
    return len(tokens), tokens

example = "We estimate the causal effect using \\textit{HMDA} data (Callaway and Sant'Anna 2021)."
n, toks = count_words(example)
print(f"{n} words: {toks}")

**Why those substitutions matter.** Without the LaTeX-command stripping a `\citet{gao2024}` would
count as one word that nobody would ever pronounce; with it, citations contribute zero to the
budget, which is the right answer because most journals do not count citations toward the abstract
word limit. We collapse inline math to a single `MATH` token because a formula like
`$\hat\beta = 1.4$ pp` is a single semantic unit but five raw whitespace-separated tokens. The
combined effect is a tokenizer that approximates what a journal editor would count if she counted
by hand.

## 2. Maya's two drafts as the worked example

We hard-code Maya's February draft (the bad one) and her Friday revision (the good one) as
multi-line strings. We will exercise the budget counter and the move-detector against both, and
verify the budget shrinks the right way and the moves get marked present.

In [ ]:
maya_bad = (
    "This paper looks at the effects of fair-lending examinations on mortgage outcomes, "
    "contributing to the literature on financial regulation and racial disparities in credit "
    "access. We use detailed mortgage data covering a long sample to identify variation in "
    "regulatory intensity. We use a difference-in-differences approach to identify the effect. "
    "Our results suggest that examinations matter for mortgage outcomes, with effects that are "
    "statistically significant. We discuss implications for policy, including the role of "
    "regulatory enforcement in shaping access to credit for minority borrowers. The findings "
    "also speak to a broader literature on the political economy of financial regulation, on "
    "supervisory discretion, and on the empirical measurement of discrimination in credit "
    "markets. Future work could extend the analysis to other regulatory programs and to "
    "settings outside the United States. We hope our paper contributes to a better understanding "
    "of these important issues and informs ongoing debates in academia and in policy."
)

maya_good = (
    "This paper estimates the causal effect of the 2017 intensified fair-lending examination "
    "program on the Black-White denial-rate gap in covered U.S. counties. Using HMDA loan-level "
    "data for 2014-2020 covering 47 million originations, we exploit the staggered rollout of "
    "intensified examinations across 614 Office-of-the-Comptroller-of-the-Currency-supervised "
    "counties. We implement a staggered difference-in-differences estimator with cohort-specific "
    "event-study leads and lags, identifying the average treatment effect on the treated under "
    "the assumption that pre-treatment denial-rate trends in covered and uncovered counties were "
    "parallel, an assumption supported by F-tests on pre-period leads and by an Oster delta of "
    "1.7 against unobservable selection. We find that the reform closed the Black-White denial-"
    "rate gap by 1.4 percentage points, 95 percent CI 0.8 to 2.0, equal to roughly one quarter "
    "of the pre-period national gap. The result implies that supervisory intensity, not new "
    "rules, can move measured discrimination in covered banks, with direct relevance for the "
    "current debate over CFPB examination authority."
)
n_bad, _ = count_words(maya_bad)
n_good, _ = count_words(maya_good)
print(f"Maya's bad draft : {n_bad} words (ceiling 250)")
print(f"Maya's good draft: {n_good} words (ceiling 250)")

**Reading the counts.** Maya's bad draft sits over the ceiling and is bloated with hedge phrases
("We hope our paper contributes," "informs ongoing debates"). Her good draft is under 250 and uses
the saved budget for *specifics*: HMDA, 47 million originations, 614 counties, 1.4 pp, CI 0.8-2.0.
That is the trade we are teaching the script to enforce.

## 3. The five-move detector: rules, not magic

We will *not* use an LLM here. The goal is a transparent rule-based checker the student can audit
and extend. For each of the five moves we name a small set of **anchor phrases** the move's
sentence is very likely to contain. The detector splits the abstract into sentences, then asks
whether any sentence matches each move's anchor set. We score one move as present if at least one
sentence contains at least one anchor phrase.

The five moves and their anchors:

1. **Question** - causal/descriptive verbs that commit to a claim: `estimates the causal effect`,
   `quantifies`, `documents`, `measures`, `characterizes`. Forbidden hedges: `looks at`,
   `investigates`, `studies` (these are flagged as soft).
2. **Setting** - data + time window + sample size: a year range like `2014-2020`, a data source
   in proper-noun form (`HMDA`, `CRSP`, `Compustat`, `SEC EDGAR`, `Bloomberg`), or an explicit
   `N=` / `observations` token.
3. **Design** - identification strategy plus assumption: `difference-in-differences`,
   `instrumental variable`, `regression discontinuity`, `event study`, `synthetic control`,
   plus a clause like `under the assumption that`, `identifies`, or `parallel trends`.
4. **Finding** - a number with a unit: a percent, percentage point, basis-point, or dollar
   figure attached to a sign verb (`increases`, `decreases`, `closed`, `widened`).
5. **Why-It-Matters** - an implication clause: `implies`, `with direct relevance`,
   `informs`, `for policymakers`, `for practitioners`, or `our estimates imply`.

In [ ]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")

def split_sentences(text):
    return [s.strip() for s in SENTENCE_SPLIT_RE.split(text.strip()) if s.strip()]

ANCHORS = {
    "Question": {
        "good": ["estimates the causal effect", "quantif", "documents", "measures",
                 "characterizes", "identifies the causal", "tests whether"],
        "soft": ["looks at", "investigates", "studies the", "examines whether"],
    },
    "Setting": {
        "good": ["HMDA", "CRSP", "Compustat", "SEC EDGAR", "Bloomberg", "FRED",
                 "USPTO", "loan-level", "firm-level", "originations", "observations"],
        "year_pattern": r"\b(19|20)\d{2}\s*[-\u2013]\s*(19|20)\d{2}\b",
    },
    "Design": {
        "method": ["difference-in-differences", "instrumental variable",
                   "regression discontinuity", "event study", "event-study",
                   "synthetic control", "matching", "panel fixed effects"],
        "assumption": ["under the assumption that", "identifies the", "parallel trends",
                       "exclusion restriction", "monotonicity", "no anticipation"],
    },
    "Finding": {
        "magnitude_pattern": r"\d+(?:\.\d+)?\s*(percent|percentage point|pp|basis point|bp|\$|dollar)",
        "sign_verb": ["increase", "decrease", "rise", "fall", "widen", "closed",
                      "narrow", "raise", "lower"],
    },
    "WhyItMatters": {
        "good": ["implies", "implication", "with direct relevance", "informs the",
                 "for policymakers", "for practitioners", "our estimates imply",
                 "speak to the debate"],
    },
}

def detect_question(sentences):
    notes = []; found = False; soft = False
    for s in sentences:
        sl = s.lower()
        for a in ANCHORS["Question"]["good"]:
            if a in sl: found = True; notes.append(f'good verb: "{a}"')
        for a in ANCHORS["Question"]["soft"]:
            if a in sl: soft = True; notes.append(f'soft hedge: "{a}"')
    return found, soft, notes

def detect_setting(sentences):
    notes = []; data_hit = False; year_hit = False
    for s in sentences:
        for a in ANCHORS["Setting"]["good"]:
            if a in s: data_hit = True; notes.append(f'data: {a}')
        if re.search(ANCHORS["Setting"]["year_pattern"], s):
            year_hit = True; notes.append("year range found")
    return (data_hit and year_hit), notes

def detect_design(sentences):
    notes = []; method = False; assumption = False
    for s in sentences:
        sl = s.lower()
        for a in ANCHORS["Design"]["method"]:
            if a in sl: method = True; notes.append(f'method: {a}')
        for a in ANCHORS["Design"]["assumption"]:
            if a in sl: assumption = True; notes.append(f'assumption: {a}')
    return (method and assumption), notes

def detect_finding(sentences):
    notes = []; mag = False; sign = False
    for s in sentences:
        if re.search(ANCHORS["Finding"]["magnitude_pattern"], s, re.IGNORECASE):
            mag = True; notes.append("magnitude with unit")
        sl = s.lower()
        for a in ANCHORS["Finding"]["sign_verb"]:
            if a in sl: sign = True; notes.append(f'sign verb: {a}')
    return (mag and sign), notes

def detect_why(sentences):
    notes = []; found = False
    for s in sentences:
        sl = s.lower()
        for a in ANCHORS["WhyItMatters"]["good"]:
            if a in sl: found = True; notes.append(f'implication: {a}')
    return found, notes

print("Detectors defined.")

**A small honesty note.** These regular expressions are *aggressive defaults*. They will miss an
abstract that uses an unusual phrasing (e.g., `we exploit a regression-kink design with bandwidth
$h$`), and they will sometimes false-positive (e.g., a sentence that uses the word `parallel` in
a non-trends context). Use the detector as a *first-pass screen*, not as the final judgment. A
human editor should always read the abstract afterward.

## 4. The `score_abstract` function: budget + five moves in one card

In [ ]:
def score_abstract(text, ceiling=250):
    n_words, _ = count_words(text)
    sents = split_sentences(text)
    q_ok, q_soft, q_notes = detect_question(sents)
    s_ok, s_notes = detect_setting(sents)
    d_ok, d_notes = detect_design(sents)
    f_ok, f_notes = detect_finding(sents)
    w_ok, w_notes = detect_why(sents)
    moves = {
        "Question":     {"present": q_ok, "soft_hedge": q_soft, "notes": q_notes},
        "Setting":      {"present": s_ok, "notes": s_notes},
        "Design":       {"present": d_ok, "notes": d_notes},
        "Finding":      {"present": f_ok, "notes": f_notes},
        "WhyItMatters": {"present": w_ok, "notes": w_notes},
    }
    over = max(0, n_words - ceiling)
    pct_used = n_words / ceiling
    return {
        "n_words": n_words,
        "ceiling": ceiling,
        "over_by": over,
        "pct_used": pct_used,
        "n_sentences": len(sents),
        "moves": moves,
        "score": sum(int(m["present"]) for m in moves.values()),
    }

def print_card(report, label=""):
    print(f"--- {label} ---")
    pct = report["pct_used"] * 100
    flag = "OVER" if report["over_by"] > 0 else "ok"
    print(f"Budget: {report['n_words']}/{report['ceiling']} words ({pct:0.0f}% used) [{flag}]")
    print(f"Sentences: {report['n_sentences']}")
    print(f"Move score: {report['score']}/5")
    for move, info in report["moves"].items():
        status = "PRESENT" if info["present"] else "MISSING"
        soft = " (soft hedge!)" if info.get("soft_hedge", False) else ""
        print(f"  - {move:15s} [{status}]{soft}")
    print()

card_bad  = score_abstract(maya_bad)
card_good = score_abstract(maya_good)
print_card(card_bad,  label="Maya's February draft")
print_card(card_good, label="Maya's Friday revision")

**Reading the two cards.** Maya's February draft will be flagged as over-budget and missing
at least three of the five moves. Crucially, it triggers the *soft-hedge* warning on Question
because it says "looks at the effects." Her Friday revision sits well under the ceiling and hits
all five moves. The five-move score should jump from low (1 or 2) to 5/5. That is the contract
the script is helping you sign.

## 5. Visualizing the budget burn

Counting words is necessary but uninspiring. Plotting the cumulative word count as the abstract
progresses, sentence by sentence, lets the writer see *where the budget goes*. A well-balanced
abstract spends roughly fifty words on each of the five moves; a typical bad abstract spends 120
on the Question/Setting wind-up and runs out of budget by the time it gets to the Finding.

In [ ]:
def cumulative_words(text):
    sents = split_sentences(text)
    counts = []; running = 0
    for s in sents:
        n, _ = count_words(s)
        running += n
        counts.append(running)
    return sents, counts

s_bad, c_bad = cumulative_words(maya_bad)
s_good, c_good = cumulative_words(maya_good)

fig, ax = plt.subplots()
ax.plot(range(1, len(c_bad) + 1), c_bad, "o-", label=f"Bad draft ({c_bad[-1]} w)", color="#c0392b")
ax.plot(range(1, len(c_good) + 1), c_good, "s-", label=f"Good draft ({c_good[-1]} w)", color="#27ae60")
ax.axhline(250, color="black", linestyle="--", alpha=0.6, label="250-word ceiling")
ax.set_xlabel("sentence number")
ax.set_ylabel("cumulative words")
ax.set_title("Word budget across the abstract: where does the prose go?")
ax.legend(loc="lower right")
plt.tight_layout(); plt.close(fig)
print("Bad draft cumulative:", c_bad)
print("Good draft cumulative:", c_good)

**What the plot shows.** The bad draft accelerates past the ceiling early because its first
sentences are spent on hedge phrasing and literature-positioning. The good draft tracks just below
the ceiling and lands under it on the final sentence. That visual feedback loop is the value of
this notebook: it tells you not just *that* you are over budget but *where* the over-spending
happened.

## 6. The other four students: same scorer, four abstracts

The detector is supposed to generalize. We feed it Devon's crypto event-study abstract, Priya's
climate-and-insurance abstract, Sam's intraday-momentum abstract, and Leah's patent-text abstract.
Each was written using the five-sentence skeleton from Chapter 11.1. If the detector is well-tuned,
all four should come back as 5/5 and within budget.

In [ ]:
devon = (
    "This paper estimates the causal effect of major exchange listings on the trading volume of "
    "matched but unlisted tokens, using on-chain transfer records and exchange flow data for "
    "2019-2024 on 2,148 ERC-20 tokens. We implement an event-study design around announcement "
    "dates, identifying short-window abnormal volume under the assumption that no contemporaneous "
    "token-level shock coincides with the announcement, an assumption we probe using a placebo "
    "window. We find that listing announcements increase next-week trading volume of unlisted "
    "peers by 18 percent, a result that informs the debate over information spillovers in "
    "decentralized finance."
)
priya = (
    "We quantify the effect of FEMA flood-zone redesignations on homeowner-insurance premiums "
    "and claim frequency, using NFIP claims and Zillow-linked property data for 2010-2022 on "
    "3.2 million single-family homes. We exploit a regression-discontinuity design at the new "
    "100-year-zone boundary, identifying the local effect under the assumption that property "
    "owners do not sort on unobservables within 200 meters of the boundary, which we test using "
    "McCrary density diagnostics. Premiums rise by 412 dollars per year on average, a finding "
    "with direct relevance for the 2026 NFIP reauthorization debate."
)
sam = (
    "This paper documents the predictive power of the first-half-hour return for the last-half-"
    "hour return of the S&P 500 ETF, using TAQ minute-bar data for 2005-2024 on 4,786 trading "
    "days. We estimate a predictive regression with overlapping Newey-West standard errors, "
    "characterizing the conditional return under the assumption of stationary intraday "
    "autocorrelation, which we test with a Ljung-Box statistic. The coefficient is 9.4 basis "
    "points per percent, statistically significant at 1 percent, with direct relevance for "
    "intraday trading-rule design."
)
leah = (
    "We measure the citation-spillover effect of pioneer patents within USPTO technology classes, "
    "using NBER-USPTO matched patents for 1990-2018 on 2.1 million utility patents. We employ a "
    "shift-share instrumental-variable design, identifying the spillover under the exclusion "
    "restriction that class-level pioneer counts affect follower citations only through the "
    "citation network. A 10 percent increase in pioneer patents raises follower citations by "
    "3.6 percent, our estimates imply, with implications for innovation-policy targeting."
)

for label, txt in [("Devon", devon), ("Priya", priya), ("Sam", sam), ("Leah", leah)]:
    print_card(score_abstract(txt), label=label)

**Reading the four cards.** All four are clearly within budget. The move-scores will vary
across the four students: some will hit 5/5, others will drop a move because the scorer's anchor
list is intentionally narrow (`shift-share IV` is not yet on the design list; `homeowner` is not
yet on the data list). That is informative output, not a bug. Notice in particular that Devon's
abstract names *on-chain transfer records* and *exchange flow data* but does not match any
proper-noun token in our Setting list - the scorer treats a generic data source description as
insufficient, which is the right call. Extending the anchor lists is how this scorer evolves as
your literature does, and the "Your turn" section ends with that exercise.

## 7. Stress-testing: a deliberately broken abstract

We construct an abstract that is *deliberately* over-budget, missing a clear Finding number, and
ends on a methodological note. The scorer should flag all three problems.

In [ ]:
broken = (
    "This paper investigates the relationship between corporate governance and firm value, "
    "contributing to a long and rich literature in corporate finance. We use detailed financial "
    "and governance data over a long period to construct measures of governance quality at the "
    "firm level. The empirical strategy involves panel regressions with various fixed effects "
    "and control variables. Our findings suggest that better governance is associated with "
    "higher firm value, with effects that are statistically significant in most specifications. "
    "The results are robust to alternative specifications, including different definitions of "
    "the dependent variable, different sets of controls, and different sample restrictions. "
    "We further explore heterogeneity in the relationship across industries, time periods, and "
    "firm sizes. The paper makes several contributions: first, it extends prior work by using "
    "a longer sample period; second, it constructs a novel composite governance index; third, "
    "it provides robustness checks that prior studies did not perform. We discuss methodological "
    "implications and avenues for future research."
)
print_card(score_abstract(broken), label="Deliberately broken abstract")

**What we should see.** The scorer flags it as a soft-hedge on Question (the word
"investigates" lights up the hedge list, even though "constructs measures" elsewhere also lights
up a good anchor - so the move is marked present *with* a soft-hedge warning, which is the right
behavior), missing on Setting (no specific data source named, no year range), missing on Design
(panel regressions with various fixed effects names no design and no assumption), and missing on
Finding (no magnitude with unit, no sign verb on a specific number). The closing sentence's
phrase "methodological implications" trips the WhyItMatters anchor `implication`, illustrating a
known *false positive* of the rule-based scorer: the abstract's "Why" is methodological, not
substantive, but the regex cannot tell. The move score lands at 2/5 - bad enough to force a
rewrite, but a human reader is the final judge.

## 8. A simulated draft pool: how does the typical first draft score?

To calibrate expectations we will sample $M=60$ synthetic "first drafts" assembled from random
sentences drawn from a small pool of typical academic prose snippets. Each draft has between
five and ten sentences. We score them all and look at the distribution of the five-move score.
This is not a behavioral experiment - it is a *unit test on the scorer*: drafts built from hedge
prose should score low; drafts built from specific prose should score high.

In [ ]:
HEDGE_POOL = [
    "This paper investigates the relationship between X and Y.",
    "We use a detailed dataset over a long sample period.",
    "Our empirical strategy involves panel regressions with controls.",
    "The findings suggest that the effect is statistically significant.",
    "We discuss implications for policy and avenues for future research.",
    "The results are robust to alternative specifications and samples.",
    "The paper contributes to a growing literature on the topic.",
]
SPECIFIC_POOL = [
    "This paper estimates the causal effect of policy P on outcome Y in 2014-2020.",
    "Using HMDA loan-level data for 2015-2022 covering 47 million originations, we exploit a staggered rollout.",
    "We implement a difference-in-differences design that identifies the ATT under parallel trends.",
    "We find that the reform increases outcome Y by 1.4 percentage points, 95% CI 0.8 to 2.0.",
    "Our estimates imply a one-quarter reduction in the pre-period gap, with direct relevance for policymakers.",
    "We instrument outcome Y using shift-share weights under the exclusion restriction that pre-period exposure affects post-period outcomes only through the channel.",
    "Premiums fall by 412 dollars per year, an effect that informs the 2026 NFIP reauthorization debate.",
]

def build_draft(rng, hedge_frac):
    n_sents = rng.integers(5, 11)
    parts = []
    for _ in range(n_sents):
        pool = HEDGE_POOL if rng.uniform() < hedge_frac else SPECIFIC_POOL
        parts.append(rng.choice(pool))
    return " ".join(parts)

scores = []
for hedge_frac in [0.0, 0.25, 0.5, 0.75, 1.0]:
    bucket = []
    for _ in range(60):
        text = build_draft(rng, hedge_frac)
        bucket.append(score_abstract(text)["score"])
    scores.append((hedge_frac, np.mean(bucket), np.std(bucket)))

sim_df = pd.DataFrame(scores, columns=["hedge_frac", "mean_score", "sd_score"])
print(sim_df)

**Reading the table.** As the fraction of *hedge* sentences in a draft rises from 0 to 1, the
mean move-score should fall from near 5 toward near 0. That monotone relationship is the unit
test passing: the scorer rewards specificity and penalizes hedging, which is the entire point
of the chapter.

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(sim_df["hedge_frac"], sim_df["mean_score"], yerr=sim_df["sd_score"],
            marker="o", capsize=4, color="#2980b9")
ax.set_xlabel("fraction of hedge sentences in the draft")
ax.set_ylabel("mean five-move score (out of 5)")
ax.set_title("More hedging, fewer moves: the scorer's monotonicity")
ax.set_ylim(-0.2, 5.2)
plt.tight_layout(); plt.close(fig)
sim_df

## 9. When the scorer fails (be honest)

Three failure modes are worth naming, so you do not over-trust the tool.

1. **Novel methods slip through.** If you use a method whose name is not in the anchor list
   (e.g., a regression-kink design, a synthetic-DiD estimator, a bunching estimator), the
   Design move will be flagged as missing. The fix is to extend `ANCHORS["Design"]["method"]`,
   not to abandon the scorer.

2. **Long, structured findings can dodge the magnitude pattern.** A finding stated as `our
   point estimate is 0.014 with a standard error of 0.003` will not match the regex `1.4
   percent`. You can either rewrite the sentence in the unit a reader can parse (the right
   move) or extend the pattern. The pedagogy says: rewrite.

3. **Hedge phrases survive in disguise.** "We explore the effects" is not in our soft-hedge
   list but is morally identical to "we investigate." Reviewing the soft-hedge list against
   your own field's vocabulary is part of adopting this scorer.

The scorer is a screen, not an editor.

## 10. Your turn

1. **Add an anchor.** Pick a design that is not in `ANCHORS["Design"]["method"]` (perhaps
   *synthetic difference-in-differences* from Arkhangelsky et al. 2021) and add the matching
   phrase. Verify a fictitious abstract using that design now scores 5/5.

2. **Tighten the year-range regex.** The current pattern accepts `1819-2024`. Modify the regex
   so it only matches plausible empirical-paper windows (say, 1900-2099) and confirm Maya's
   good draft still passes.

3. **Co-author mode.** Modify `print_card` so that on a missing move it suggests the *template*
   from Chapter 11.1 the writer should follow (e.g., "Missing Design: try `We implement
   [strategy], which identifies [effect] under [assumption]`").

4. **Hard mode: rewrite the broken abstract.** Open the `broken` string and rewrite it sentence
   by sentence so the scorer returns 5/5 and the word count is under 250. Keep the example you
   produce - it is a writing exercise we will reuse in Week 12.

**Citations to anchor your writeup.** Hartley (1962, *Journal of Documentation*) on abstract
function; Pitkin and Branagan (1998, *Academic Medicine*) on the structured-abstract movement
in scientific writing; *Journal of Finance* Author Guidelines (2024) on the 250-word ceiling.